1.消息类型

在langchain中，发送给LLM的消息、LLM返回的消息都是统一被封装成BaseMessage，它是Agent中基本的上下文单元。

在LangChain中，我们不需要自己创建BaseMessage对象，LangChain提供了一些常用的消息类型，如HumanMessage、SystemMessage、AIMessage。

In [16]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 定义工具
@tool
def get_weather(location: str) -> str:
    """Get the weather in a given location."""
    return f"current weather in {location} is cloudy"

# 创建Agent
agent = create_agent(model = "deepseek-v4-pro", tools = [get_weather])

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        # {"role": "system", "content": "你是一个热心的AI助手"},
        # {"role": "user", "content": "你好，哈哈哈我是强子"},
        # {"role": "assistant", "content": "你好，强子，很开心人认识你"},
        # {"role": "user", "content": "现在上海天气如何"},
        SystemMessage(content = "你是一个热心的AI助手"),
        HumanMessage(content = "你好，哈哈哈我是强子"),
        AIMessage(content = "你好，强子，很开心认识你"),
        HumanMessage(content = "现在上海天气如何")
    ]
})

print(response)

{'messages': [SystemMessage(content='你是一个热心的AI助手', additional_kwargs={}, response_metadata={}, id='2329a256-c96c-4222-9125-b7c181089afc'), HumanMessage(content='你好，哈哈哈我是强子', additional_kwargs={}, response_metadata={}, id='d981cc9c-e95f-4411-8181-763537ade956'), AIMessage(content='你好，强子，很开心认识你', additional_kwargs={}, response_metadata={}, id='1fcabc3c-910e-4827-80f6-a75aedfd6d7f', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='现在上海天气如何', additional_kwargs={}, response_metadata={}, id='4cb92ac1-a1f9-4618-b84e-1f40bb19454b'), AIMessage(content='', additional_kwargs={'refusal': None, 'reasoning_content': '用户想知道上海现在的天气。让我调用天气查询工具来获取上海的最新天气。\n\n用户提到了上海，我需要将"上海"作为location参数传递。'}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 303, 'total_tokens': 379, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 31, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cac

In [17]:
for message in response["messages"]:
    print(message.pretty_print())    # 美观打印

================================ System Message ================================

你是一个热心的AI助手
None
================================ Human Message =================================

你好，哈哈哈我是强子
None
================================== Ai Message ==================================

你好，强子，很开心认识你
None
================================ Human Message =================================

现在上海天气如何
None
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_PuHCbSNs4gW6KwY460956113)
 Call ID: call_00_PuHCbSNs4gW6KwY460956113
  Args:
    location: 上海
None
================================= Tool Message =================================
Name: get_weather

current weather in 上海 is cloudy
None
================================== Ai Message ==================================

强子，现在上海的天气是**多云**☁️。出门的话可以不用带伞，但如果你想以防万一，带把伞也不碍事哈。有啥需要帮忙的随时叫我！
None


2.多模态消息

2.1 在线图片

In [62]:
from langchain.chat_models import init_chat_model
import os
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 初始化模型
model = init_chat_model(
    model = "qwen3.7-plus",  # 多模态模型
    model_provider = "openai",
    base_url = "https://ws-3ssufixwbthuxno3.cn-beijing.maas.aliyuncs.com/compatible-mode/v1",
    api_key = os.getenv("DASHSCOPE_API_KEY")
)

In [63]:
# 创建智能体Agent
agent = create_agent(model = model)

In [64]:
# 准备多模态消息
# message = {
#     "role": "user",
#     "content": [
#         {"role": "text", "text": "描述一下这个图片的内容"},
#         {"role": "image", "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg "},
#     ]
# }

message = HumanMessage(
    [
        {"role": "text", "text": "描述一下这个图片的内容"},
        {"role": "image_url", "image_url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
    ])

In [67]:
stream = agent.stream(
    {
    "messages": [message]
    },
    stream_mode = "messages"
)

for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end = "", flush = True)

BadRequestError: data: {"error":{"code":"invalid_parameter_error","param":null,"message":"if content is list. item must be dict and key[type] should in dict","type":"invalid_request_error"},"id":"chatcmpl-312db507-19c0-945c-b2d8-f02e76f50493"}